In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

**Контекст.** Признаки, построенные на фильтрах, — это ответы на вопросы к *истории*
сущности: встречалась ли она во фроде (черный список), встречалась ли вообще (новизна),
встречалась ли в данном контексте (контекстная новизна), накоплена ли у нее чистая история
(белый список). Элемент фильтра — ключ сущности; от выбора ключа зависит и сила сигнала,
и устойчивость к ложным срабатываниям фильтра.

# 0. Загрузка данных

In [2]:
df_transaction = pd.read_csv('../data/train_transaction.csv')
df_identity = pd.read_csv('../data/train_identity.csv')
df_train = df_transaction.merge(df_identity, on='TransactionID', how='left')

df_train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df_transaction = pd.read_csv('../data/test_transaction.csv')
df_identity = pd.read_csv('../data/test_identity.csv')
df_test = df_transaction.merge(df_identity, on='TransactionID', how='left')

df_test

,TransactionID,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id-31,id-32,id-33,id-34,id-35,id-36,id-37,id-38,DeviceType,DeviceInfo
0,3663549,18403224,31.950,W,10409,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3663550,18403263,49.000,W,4272,111.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3663551,18403310,171.000,W,4476,574.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3663552,18403310,284.950,W,10989,360.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3663553,18403317,67.950,W,18018,452.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506686,4170235,34214279,94.679,C,13832,375.0,185.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
506687,4170236,34214287,12.173,C,3154,408.0,185.0,mastercard,224.0,debit,...,chrome 43.0 for android,NaN,NaN,NaN,F,F,T,F,mobile,ALE-L23 Build/HuaweiALE-L23
506688,4170237,34214326,49.000,W,16661,490.0,150.0,visa,226.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
506689,4170238,34214337,202.000,W,16621,516.0,150.0,mastercard,224.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df_train['isTrain'] = True
df_test['isTrain'] = False
df_test = df_test.rename(columns=lambda c: c.replace("id-", "id_", 1) if c.startswith("id-") else c)

df = pd.concat([df_train, df_test], axis=0, ignore_index=True)

df

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,isTrain
0,2987000,0.0,86400,68.500,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1,2987001,0.0,86401,29.000,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
2,2987002,0.0,86469,59.000,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
3,2987003,0.0,86499,50.000,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,2987004,0.0,86506,50.000,H,4497,514.0,150.0,mastercard,102.0,...,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1097226,4170235,NaN,34214279,94.679,C,13832,375.0,185.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1097227,4170236,NaN,34214287,12.173,C,3154,408.0,185.0,mastercard,224.0,...,NaN,NaN,NaN,F,F,T,F,mobile,ALE-L23 Build/HuaweiALE-L23,False
1097228,4170237,NaN,34214326,49.000,W,16661,490.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1097229,4170238,NaN,34214337,202.000,W,16621,516.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


# 1. Ключи сущностей

**Форма ключа — строка со значениями компонентов через разделитель `|`.**

Технические требования к сборке ключа:

* **разделитель обязателен** — иначе `card1=12, addr1=345` и `card1=123, addr1=45`
  дадут один ключ
* **все числовые компоненты приводятся к `int`** перед форматированием: чтобы избежать лишних `.0` в ключе
* **ключ не определен, если пуст хотя бы один компонент** (маркер `ND`) (альтернатива —
  подставдение в ключ фиксированного токена вместо пустого компонента — склеивает разные сущности в одну,
  добавляя семантические коллизии поверх коллизий фильтра)

**Следствие для признаков:** в силу последнего пункта любой признак, построенный поверх ключей, будет принимать одно из трех состояний:
* «ключ не определен» /
* «в фильтре»
* «не в фильтре».

Кодировать такие признаки будем как числовые с NaN (нативная обработка пропусков от CatBoost). Схлопывать «не определен» и «не в фильтре» в один класс нельзя:
модель выучила бы «нет данных = чисто», а для `device_id` таких строк 87%.

In [5]:
df['D1Norm'] = df['D1'] - (df['TransactionDT'] // 86400)

In [6]:
def get_cid(card1):  # card1-id
    if np.isnan(card1):
        return "ND"
    return f"{int(card1)}"

def get_aid(addr1):  # addr1-id
    if np.isnan(addr1):
        return "ND"
    return f"{int(addr1)}"

def get_caid(card1, addr1):  # card1-addr1-id
    if np.isnan(card1) or np.isnan(addr1):
        return "ND"
    return f"{int(card1)}|{int(addr1)}"

def get_uid(card1, addr1, d1_norm):  # card1-addr1-d1_norm-id
    if np.isnan(card1) or np.isnan(addr1) or np.isnan(d1_norm):
        return "ND"
    return f"{int(card1)}|{int(addr1)}|{int(d1_norm)}"

def get_device_id(device_info, id_31, id_33):  # DeviceInfo-id_31-id_33-id
    if pd.isna(device_info) or pd.isna(id_31) or pd.isna(id_33):
        return "ND"
    return f"{device_info}|{id_31}|{id_33}"

def get_pair(base_key, ctx):  # связка "сущность|контекст"
    if base_key == "ND" or pd.isna(ctx):
        return "ND"
    return f"{base_key}|{ctx}"

In [7]:
df['cid'] = [get_cid(card1) for card1 in df.card1]
df['aid'] = [get_aid(addr1) for addr1 in df.addr1]
df['caid'] = [get_caid(card1, addr1) for card1, addr1 in zip(df.card1, df.addr1)]
df['uid'] = [get_uid(card1, addr1, d1_norm) for card1, addr1, d1_norm in zip(df.card1, df.addr1, df.D1Norm)]
df['device_id'] = [get_device_id(device_info, id_31, id_33) for device_info, id_31, id_33 in zip(df.DeviceInfo, df.id_31, df.id_33)]

# ключи-связки: сущность-плательщик + контекст платежа
df['uid_email']  = [get_pair(u, e) for u, e in zip(df.uid, df.P_emaildomain)]
df['uid_device'] = [get_pair(u, d) for u, d in zip(df.uid, df.device_id)]
df['cid_email']  = [get_pair(c, e) for c, e in zip(df.cid, df.P_emaildomain)]

## 2. Базовая статистика по ключам

Метрики, определяющие пригодность ключа:

* **Доля транзакций с определенным ключом**
* **Кардинальность** (влияет на размеры фильтра)
* **Число транзакций на сущность** (медиана, среднее, доля одноразовых) — «полезная емкость»
  обоих семейств признаков;
* **Доля фродовых сущностей** — доля сущностей, засветившихся во фроде
* **Переносимость в test** — доля тестовых строк, чей ключ встречался в train

In [8]:
def key_stats(df, key_name):
    k = df[key_name]
    defined = k != 'ND'
    tr, te = defined & df.isTrain, defined & ~df.isTrain

    vc_tr = k[tr].value_counts()          # транзакций на сущность (train)
    ents_tr = set(vc_tr.index)            # все сущности train
    fraud_ents = set(k[tr & (df.isFraud == 1)])   # сущности, засветившиеся во фроде

    return {
        'key': key_name,
        'Доля определенных': round(defined.mean(), 3),
        'Кардинальность (train)': len(ents_tr),
        'Медиана числа транзакций на сущность': vc_tr.median(),
        'Среднее числа транзакций на сущность': round(vc_tr.mean(), 2),
        'Доля одноразовых сущностей': round((vc_tr == 1).mean(), 3),
        'Доля фродовых сущностей': round(len(fraud_ents) / len(ents_tr), 4),
        'Количество test-строк с ключом из train': round(k[te].isin(ents_tr).mean(), 3),
        'Количество test-строк с ключом из фрод-множества': round(k[te].isin(fraud_ents).mean(), 4),
    }

In [9]:
KEYS = ['cid', 'aid', 'caid', 'uid', 'device_id', 'cid_email', 'uid_email', 'uid_device']

stats = pd.DataFrame([key_stats(df, key) for key in KEYS]).set_index('key')

stats.T.map(lambda v: f'{v:g}') 

key,cid,aid,caid,uid,device_id,cid_email,uid_email,uid_device
Доля определенных,1,0.88,0.88,0.875,0.127,0.851,0.729,0.875
Кардинальность (train),13553,332,37531,199070,2923,38162,207974,216884
Медиана числа транзакций на сущность,4,3,2,1,2,2,1,1
Среднее числа транзакций на сущность,43.57,1580.83,13.98,2.63,24.33,13,2.08,2.41
Доля одноразовых сущностей,0.254,0.37,0.395,0.58,0.435,0.415,0.658,0.63
Доля фродовых сущностей,0.1284,0.259,0.0813,0.0242,0.1232,0.076,0.022,0.0233
Количество test-строк с ключом из train,0.982,1,0.939,0.244,0.322,0.933,0.209,0.236
Количество test-строк с ключом из фрод-множества,0.756,0.9973,0.5021,0.0052,0.1805,0.568,0.003,0.0045


## Выводы по базовым метрикам

Ключи `cid`,`caid`, `uid` выстраиваются в градиент по гранулярности: $13.5 \rightarrow 37.5 \rightarrow 199$ (тыс. сущностей).

**Гранулярность ключа** — насколько мелко он дробит популяцию: грубый ключ объединяет
под одним значением много разных плательщиков (`cid` — BIN-подобная группа карт),
гранулярный приближается к отдельной сущности (`uid` ≈ конкретная карта конкретного
клиента). Численно выражается кардинальностью и числом транзакций на сущность.


При этом с ростом гранулярности падает доля фродовых сущностей: $0.128 \rightarrow 0.081 \rightarrow 0.024$.

**Почему так происходит?** - Сущность считается фродовой, если
*хотя бы одна* ее транзакция оказалась фродовой. Чем больше транзакций приходится
на сущность, тем выше шанс, что среди них попадется фродовая: у `cid` в среднем
десятки транзакций на сущность, у `uid` — единицы. Как показано ниже, гранулярность управляет и силой сигнала, и хрупкостью признака
к ошибкам фильтра.

`aid` ($332$ сущности) и `device_id` ($2.9$ тыс. при покрытии $12.7%$) выбиваются из градиента — разбор далее.

Ключи-связки `cid_email`, `uid_email`, `uid_device` — сущность-плательщик в конкретном
контексте платежа (домен почты, устройство). Они гранулярнее своих базовых ключей и
используются для признака «знакомая сущность в новом контексте».

# 3. Признак черного списка

Признак: «сущность замечена во фроде ранее текущей транзакции».

Строится потоково, в порядке времени, по правилу **query-then-update**: сначала запрос
к структуре (это и есть значение признака), затем обновление структуры текущей
транзакцией. Порядок гарантирует, что признак не зависит от собственной метки.

Фильтр Блума не дает ложноотрицательных ответов: для сущности, действительно
находящейся в фильтре, ответ всегда «да». Ошибки строго односторонние — сущность,
в фильтре отсутствующая, получает ответ «да» с вероятностью $\epsilon$ (False Positive
Rate фильтра). Признак зашумлен в одну сторону: 0 → 1.

Точность признака «фильтр сказал да» выражается через долю запросов, попадающих
в множество, $p$, и FPR фильтра:

$precision = \frac{p}{p + \epsilon(1 - p)}$

**Важно:** $p$ - не доля фродовых сущностей из §2, но **транзакционно-взвешенная** доля фрода, то есть доля транзакций, относящихся к фродовым сущностям. Эти величины могут расходиться в разы: крупные
сущности одновременно чаще оказываются фродовыми и чаще запрашиваются.

Поэтому сначала измерим поведение признака при $\epsilon = 0$ (точные структуры) — это
одновременно потолок качества BF-признаков и источник корректного $p$ для формулы.

## 3.1. Признак на точных структурах

In [10]:
BL_KEYS = ['cid', 'caid', 'uid']

In [11]:
df = df.sort_values('TransactionDT').reset_index(drop=True)

def simulate_bl(df, key_col):
    """Time-aware признак черного списка на точных структурах: query-then-update."""
    frauded = set()
    keys = df[key_col].to_numpy()
    fraud = df['isFraud'].to_numpy()
    is_train = df['isTrain'].to_numpy()

    bl = np.full(len(df), np.nan)
    
    for i in range(len(df)):
        k = keys[i]
        if k == 'ND':
            continue
        bl[i] = k in frauded    
        if is_train[i] and fraud[i] == 1:
            frauded.add(k)
    return bl

for key in BL_KEYS:
    df[f'bl_{key}'] = simulate_bl(df, key)

In [12]:
rows = []
tr = df.isTrain
for key in BL_KEYS:
    col = f'bl_{key}'
    m = tr & df[col].notna()
    pos = m & (df[col] == 1)
    fr1 = df.loc[pos, 'isFraud'].mean()
    neg = m & (df[col] == 0)
    fr0 = df.loc[neg, 'isFraud'].mean()
    rows.append({
        'Доля срабатываний': round(pos.sum() / m.sum(), 4),
        'Fraud rate при 1': round(fr1, 4),
        'Fraud rate при 0': round(fr0, 4),
        'Соотношение fraud rate при 1 и 0': round(fr1 / fr0, 4),
    })
pd.DataFrame(
    rows,
    index=BL_KEYS
)

,Доля срабатываний,Fraud rate при 1,Fraud rate при 0,Соотношение fraud rate при 1 и 0
cid,0.6661,0.0481,0.0088,5.4512
caid,0.3805,0.0494,0.0094,5.2659
uid,0.0233,0.6604,0.0094,70.0120


**Вывод 1: транзакционно-взвешенная доля срабатываний сильно отличается
от доли фродовых сущностей.** У `cid` признак срабатывает на $66.6%$ транзакций
при $12.8%$ фродовых сущностей из §2: фродовые `cid` — это крупные группы, на которые приходится
непропорционально много транзакций (в среднем $43.6$ транзакции на сущность против
$2.6$ у `uid`). У `uid` расхождения почти нет ($2.3%$ против $2.4%$) — сущности мелкие,
эффект отбора не проявляется.

**Вывод 2: компромисс гранулярности выглядит так:**

* **грубый ключ** (`cid`): срабатывает часто (66.6% транзакций),
  но слабо разделяет — $4.8%$ фрода против $0.9%$, lift $\approx 5.5$. Утверждение
  «в этой BIN-группе кто-то когда-то нафродил» верно для двух третей транзакций
  и потому мало что говорит о конкретной из них
* **гранулярный ключ** (`uid`): срабатывает редко ($2.3%$ транзакций), но почти
  детерминирует метку — $66%$ фрода против $0.9%$, lift $\approx 70$

Таким образом, при $\epsilon = 0$ признак `bl_uid` — исключительно сильный,
`bl_cid` — умеренный. Ниже — что с ними делает ненулевой FPR.

## 3.2. Построение признака на всех категориальных данных

Данный признак можно построить по любой категориальной колонке датасета. Однако
для большинства колонок он либо вырождается, либо дублирует то, что модель уже имеет.

### 3.2.1. Вырождение на малой кардинальности

Для колонки с единицами–десятками значений (`ProductCD`, `card4`, `card6`, `M1–M9`, почти все `id_*`) черный список за первые
недели потока наполняется целиком: во фроде побывало каждое значение, и признак
становится константой. Примером служит представленный выше ключ `aid` ($332$ значения):

In [13]:
key = 'aid'
df[f'bl_{key}'] = simulate_bl(df, key)
col = f'bl_{key}'
m = tr & df[col].notna()
pos = m & (df[col] == 1)
fr1 = df.loc[pos, 'isFraud'].mean()
neg = m & (df[col] == 0)
fr0 = df.loc[neg, 'isFraud'].mean()
pd.DataFrame(
    {
        'Доля срабатываний': round(pos.sum() / m.sum(), 4),
        'Fraud rate при 1': round(fr1, 4),
        'Fraud rate при 0': round(fr0, 4),
        'Соотношение fraud rate при 1 и 0': round(fr1 / fr0, 4),
    },
    index=['aid']
)

,Доля срабатываний,Fraud rate при 1,Fraud rate при 0,Соотношение fraud rate при 1 и 0
aid,0.9847,0.0248,0.0107,2.3141


Доля срабатываний черного списка по ключу `aid` - $0.98$, то есть признак «в этом регионе был фрод» почти всегда истинен.

### 3.2.2. Дублирование

Для колонок, которые уже подаются в модель
как категориальные (`card1`, `addr1`, `P_emaildomain`, `DeviceInfo`), черный список —
это бинарный последовательный target encoding: «была ли у значения ранее фродовая история».
CatBoost с `has_time=True` **уже строит** по каждой такой колонке ordered target
statistics — та же статистика по прошлому, только выражаемая долей фрода до транзакцией,
а не однобитновым значением. Таким образом, черный списко для подобных колонок - урезанная
версия того, что модель и так имеет. Строго говоря, это касается и черного списка по `cid`:
`card1` — категориальная колонка baseline, так что ожидается низкая важность данного признака.

Отсюда **критерий отбора ключей** для фильтров:

1. **высокая кардинальность** — иначе множество наполняется целиком
2. **персистентность** — сущность повторяется, и фрод на ней концентрируется
   (у одноразовой сущности признак всегда 0)
3. **отсутствие в модели как готовой категориальной колонки** — иначе информация
   дублирует ordered target statistics

## 3.3. Точность при ненулевом FPR

In [14]:
EPS_GRID = [0.001, 0.01, 0.05, 0.10]

In [15]:
def precision(p, eps):
    return p / (p + eps * (1 - p))

In [16]:
# транзакционно-взвешенная доля срабатываний (доля срабатываний при eps=0)
p = {}
for k in BL_KEYS:
    m = df.isTrain & df[f'bl_{k}'].notna()
    p[k] = (m & (df[f'bl_{k}'] == 1)).sum() / m.sum()

pd.DataFrame(
    [[p[k]] + [precision(p[k], e) for e in EPS_GRID] for k in BL_KEYS],
    index=BL_KEYS,
    columns=['p'] + [f'ε={e:.1%}' for e in EPS_GRID],
).round(3)

,p,ε=0.1%,ε=1.0%,ε=5.0%,ε=10.0%
cid,0.666,0.999,0.995,0.976,0.952
caid,0.380,0.998,0.984,0.925,0.860
uid,0.023,0.960,0.705,0.323,0.193


**Вывод: устойчивость к $\epsilon$ обратна силе сигнала.** Грубые ключи практически
не деградируют (даже при $\epsilon = 10\%$ точность `cid` — $0.95$), потому что их
фильтр и так покрывает большинство транзакций, и лишние срабатывания почти ничего
не добавляют. `uid` уже при $\epsilon = 5\%$ теряет две трети точности: множество редких
попаданий тонет в ложных срабатываниях по всей остальной массе транзакций.

## 3.4. Прогноз деградации признака

Если ложные срабатывания приходятся на случайные строки, ожидаемая доля фрода среди
единиц:

$precision(ε) · fr(1) + (1 − precision(ε)) · fr(0)$

In [17]:
rows = []
for k in BL_KEYS:
    m = df.isTrain & df[f'bl_{k}'].notna()
    fr1 = df.loc[m & (df[f'bl_{k}'] == 1), 'isFraud'].mean()
    fr0 = df.loc[m & (df[f'bl_{k}'] == 0), 'isFraud'].mean()
    row = {}
    for e in [0.0] + EPS_GRID:
        pr = 1.0 if e == 0 else precision(p[k], e)
        row[f'ε={e:.1%}'] = round((pr * fr1 + (1 - pr) * fr0) / fr0, 1)
    rows.append(row)

pd.DataFrame(
    rows,
    index=BL_KEYS
)  # lift = fraud rate при 1 / fraud rate при 0

,ε=0.0%,ε=0.1%,ε=1.0%,ε=5.0%,ε=10.0%
cid,5.5,5.4,5.4,5.3,5.2
caid,5.3,5.3,5.2,4.9,4.7
uid,70.0,67.2,49.6,23.3,14.3


**Вывод: даже сильно деградировавший гранулярный ключ остается информативнее грубого.**
При $\epsilon = 10\%$ lift `bl_uid` становится в $5$ раз меньше исходного, но сохраняет $14.3$ против $5.3$
у `bl_cid`. Практическое следствие для эксперимента: провала качества модели на больших
$\epsilon$ может и не случиться — деградация будет постепенной, и ее масштаб важно
оценивать не только по AUC модели, но и напрямую на уровне признаков.

Полученные значения lift - прогноз, который будет проверен эмпирически: если замеренные
значения совпадут с предполагаемыми — механизм деградации понят верно.

# 4. Признак новизны

Признак: «сущность видится впервые». Формально — инверсия ответа фильтра, содержащего
все встреченные ранее сущности, независимо от метки.

Гипотеза: свежесозданные сущности статистически рискованнее.

Кроме того, данный признак - однобитный аналог frequency encoding, намеренно исключенного из baseline.

Важное отличие от черного списка: фильтр новизны **не требует меток класса**, поэтому
продолжает наполняться и на тестовом периоде, тогда как черный список на тесте
заморожен.

## 4.1. Признак на точных структурах

In [18]:
def simulate_novelty(df, key_col):
    """Time-aware признак новизны на точных структурах: query-then-update, обновление в том числе на test."""
    seen = set()
    keys = df[key_col].to_numpy()
    nv = np.full(len(df), np.nan)

    for i in range(len(df)):
        k = keys[i]
        if k == 'ND':
            continue
        nv[i] = k not in seen     
        seen.add(k)               
    return nv

for key in BL_KEYS:
    df[f'nv_{key}'] = simulate_novelty(df, key)

In [19]:
rows = []
tr = df.isTrain
for key in BL_KEYS:
    col = f'nv_{key}'
    m = tr & df[col].notna()
    pos = m & (df[col] == 1)
    fr1 = df.loc[pos, 'isFraud'].mean()
    neg = m & (df[col] == 0)
    fr0 = df.loc[neg, 'isFraud'].mean()
    rows.append({
        'Доля срабатываний': round(pos.sum() / m.sum(), 4),
        'Fraud rate при 1': round(fr1, 4),
        'Fraud rate при 0': round(fr0, 4),
        'Соотношение fraud rate при 1 и 0': round(fr1 / fr0, 4),
    })
pd.DataFrame(
    rows,
    index=BL_KEYS
)

,Доля срабатываний,Fraud rate при 1,Fraud rate при 0,Соотношение fraud rate при 1 и 0
cid,0.0230,0.0257,0.0352,0.7293
caid,0.0715,0.0256,0.0245,1.0432
uid,0.3801,0.0203,0.0273,0.7446


Предложенная выше гипотеза не подтвердилась: новые сущности не
рискованнее виденных — например, у `uid` fraud rate среди новых даже ниже ($2.0%$ против $2.7%$).

Полученный результат согласуется с выводами о концентрации фрода на повторяющихся сущностях (§3.1): первая
транзакция сущности чаще легитимна, риск нарастает с последующими.

Кроме того, у признака практически нет предсказательной силы: соотношение fraud rate при 1 и 0 близко к единице для всех представленных ключей. 

**Вывод: признак новизны в эксперимент не включается.** Сам фильтр «виденных
сущностей» при этом остается в работе — как условие в признаке контекстной новизны.

# 5. Признак «знакомая сущность в новом контексте»

# 6. Белый список